In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import spacy
import re
import contractions

#### LOAD DATA :-

In [ ]:
with open('data.txt','r',encoding='utf-8') as file:
    data=file.read()

In [ ]:
print(data)

## TEXT NORMALIZATION :-

CONVERT ALL THE DAT TO LOWERCASE :-

In [ ]:
data=data.lower()

REMOVING THE EXTRA SPACE :-


In [ ]:
data=re.sub(r"\s{2,}","",data)
data

REMOVING THE NUMBERS :-

In [ ]:
data=re.sub(r'\d+\.','',data)
data

In [ ]:
data = re.sub(r"\d+", "", data)
data

CONCENTRACTION OF DATA :-

In [ ]:
data=contractions.fix(data)

REMOVING ALL THE PUNCHUCATION AND SPECIAL CHARACTERS :-

In [ ]:
data=re.sub(r'[^0-9a-zA-Z\s]','',data)

REMOVING THE EMOJI :-

In [ ]:
import emoji

data=emoji.replace_emoji(data)

TEXTBLOB :-

In [ ]:
from  textblob import TextBlob
value=TextBlob(data).correct()

LEMMATIZATION :-


In [ ]:

nlp=spacy.load("en_core_web_sm")
tokens=nlp(data)
updated_tokens=[token.lemma_ for token in tokens if not token.is_stop]
data=' '.join(updated_tokens).strip() #strip() is usewd to remove the extra space 


CHIUNKING:-


In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)

chunks=list(set(splitter.split_text(data)))

EMBEDDING :-


In [ ]:
embedding_model=SentenceTransformer(
    model_name_or_path='sentence-transformers/all-miniLM-L6-V2'
)
chunk_embedding=embedding_model.encode(chunks).astype('float32')

CREATION OF VECTOR_DB :-


In [ ]:
dimension=chunk_embedding.shape[1]
dimension

In [ ]:
faiss.normalize_L2(chunk_embedding)

In [ ]:
index_faiss_db=faiss.IndexFlatIP(dimension)
index_faiss_db.add(chunk_embedding)

In [ ]:
def rag_query(query,k):
    query_embeddings=embedding_model.encode_query(query).astype('float32')
    query_embeddings=query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    distamce,index=index_faiss_db.search(query_embeddings,k=k)
    
    R_chunk=[chunks[i] for i in index[0]]
    R_str=' '.join(R_chunk)
    return R_str

user_prompt='what is python'
user_prompt=re.sub(r'[^0-9a-zA-Z]','',user_prompt)

responce=rag_query(user_prompt,2)
print(responce)

In [ ]:

def r_search(query,k=3):
    query_embeddings = embedding_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance,index = index_faiss_db.search(query_embeddings,k=k)
    R_chunks = [chunks[i] for i in index[0]]
    R_str = ' '.join(R_chunks)
    return R_str
def g_text(r_search):
        import os
        import requests
    
        API_URL = "https://router.huggingface.co/v1/chat/completions"
    
        headers = {
            "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
        }
        def query(payload):
            response = requests.post(API_URL, headers=headers, json=payload)
            return response.json()
        prompt = f'''
                    You're an helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result.
            '''
        response = query({
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "model": "deepseek-ai/DeepSeek-R1:novita"
        })
    
        return response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

r_response = r_search(user_prompt)
g_response = g_text(r_response)
print(g_response)